# Signals, filters and execution

Generate Keltner breakout/mean-reversion signals, filter entries by regime and simulate fills, costs and position state.

**Part 2/4 of a shared-state notebook series.** This part uses variables/functions from earlier parts. Opening another notebook normally creates a new kernel. Use `python scripts/run_series.py all_for_one --export outputs/all_for_one_combined.ipynb` from the repository root, then open the exported notebook for interactive execution. Exporting does not execute the research or restore missing data.

**Status:** curated research source; original saved outputs preserved. Original notes about results are historical observations, not revalidated conclusions. Read `docs/DATA_AND_RUNNING.md` before execution.

Source: `ALL_FOR_ONE.ipynb`, original cells 50–131 (zero-based).


# Keltner Strategy Signal Engine

In [46]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Literal

import numpy as np
import pandas as pd

In [47]:
@dataclass(frozen=True)
class KeltnerIndicatorConfig:
    ema_period: int = 20
    atr_period: int = 20
    atr_multiplier: float = 2.0

    price_source: Literal[
        "close",
        "hlc3",
    ] = "hlc3"

    atr_smoothing: Literal[
        "rma",
        "ema",
        "sma",
    ] = "rma"

    # current:
    # signal เวลา t เปรียบเทียบกับ channel ที่คำนวณถึงเวลา t
    #
    # previous:
    # signal เวลา t เปรียบเทียบกับ channel ที่รู้มาตั้งแต่ t-1
    level_reference: Literal[
        "current",
        "previous",
    ] = "current"

In [48]:
@dataclass(frozen=True)
class KeltnerSignalConfig:
    strategy_type: Literal[
        "breakout",
        "mean_reversion",
    ]

    exit_rule: Literal[
        "basis_cross",
        "channel_reentry",
        "opposite_band",
    ] = "basis_cross"

    allow_long: bool = True
    allow_short: bool = True

In [49]:
@dataclass
class KeltnerSignalResult:
    strategy_id: str
    execution_timeframe: str

    indicator_config: KeltnerIndicatorConfig
    signal_config: KeltnerSignalConfig

    data: pd.DataFrame

In [50]:
def float_to_token(value: float) -> str:
    return (
        f"{value:g}"
        .replace("-", "neg")
        .replace(".", "p")
    )


def make_keltner_strategy_id(
    execution_timeframe: str,
    indicator_cfg: KeltnerIndicatorConfig,
    signal_cfg: KeltnerSignalConfig,
) -> str:
    return "_".join(
        [
            "keltner",
            signal_cfg.strategy_type,
            execution_timeframe,
            f"ema{indicator_cfg.ema_period}",
            f"atr{indicator_cfg.atr_period}",
            f"m{float_to_token(indicator_cfg.atr_multiplier)}",
            indicator_cfg.price_source,
            indicator_cfg.atr_smoothing,
            indicator_cfg.level_reference,
            signal_cfg.exit_rule,
        ]
    )

In [51]:
def calculate_true_range(
    bars: pd.DataFrame,
) -> pd.Series:
    prev_close = bars["close"].shift(1)

    true_range = pd.concat(
        [
            bars["high"] - bars["low"],
            (bars["high"] - prev_close).abs(),
            (bars["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    return true_range.astype(float)

In [52]:
def smooth_atr(
    true_range: pd.Series,
    period: int,
    method: str,
) -> pd.Series:
    if period <= 0:
        raise ValueError(
            "ATR period must be positive."
        )

    if method == "rma":
        return true_range.ewm(
            alpha=1.0 / period,
            adjust=False,
            min_periods=period,
        ).mean()

    if method == "ema":
        return true_range.ewm(
            span=period,
            adjust=False,
            min_periods=period,
        ).mean()

    if method == "sma":
        return true_range.rolling(
            window=period,
            min_periods=period,
        ).mean()

    raise ValueError(
        f"Unknown ATR smoothing method: {method}"
    )

In [53]:
def calculate_keltner_price_source(
    bars: pd.DataFrame,
    price_source: str,
) -> pd.Series:
    if price_source == "close":
        return bars["close"].astype(float)

    if price_source == "hlc3":
        return (
            bars["high"].astype(float)
            + bars["low"].astype(float)
            + bars["close"].astype(float)
        ) / 3.0

    raise ValueError(
        f"Unknown price source: {price_source}"
    )

In [54]:
def add_keltner_features(
    execution_bars: pd.DataFrame,
    cfg: KeltnerIndicatorConfig,
) -> pd.DataFrame:
    required = {
        "open",
        "high",
        "low",
        "close",
    }

    missing = required.difference(
        execution_bars.columns
    )

    if missing:
        raise ValueError(
            f"Missing OHLC columns: {sorted(missing)}"
        )

    if not isinstance(
        execution_bars.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "Execution bars index must be DatetimeIndex."
        )

    if not execution_bars.index.is_monotonic_increasing:
        raise ValueError(
            "Execution bars must be sorted by time."
        )

    if execution_bars.index.duplicated().any():
        raise ValueError(
            "Execution bars contain duplicate timestamps."
        )

    if cfg.ema_period <= 0:
        raise ValueError(
            "EMA period must be positive."
        )

    if cfg.atr_period <= 0:
        raise ValueError(
            "ATR period must be positive."
        )

    if cfg.atr_multiplier <= 0:
        raise ValueError(
            "ATR multiplier must be positive."
        )

    d = execution_bars.copy()

    d["kc_source"] = calculate_keltner_price_source(
        d,
        cfg.price_source,
    )

    d["kc_true_range"] = calculate_true_range(d)

    d["kc_atr"] = smooth_atr(
        d["kc_true_range"],
        period=cfg.atr_period,
        method=cfg.atr_smoothing,
    )

    d["kc_basis"] = d["kc_source"].ewm(
        span=cfg.ema_period,
        adjust=False,
        min_periods=cfg.ema_period,
    ).mean()

    d["kc_upper"] = (
        d["kc_basis"]
        + cfg.atr_multiplier * d["kc_atr"]
    )

    d["kc_lower"] = (
        d["kc_basis"]
        - cfg.atr_multiplier * d["kc_atr"]
    )

    if cfg.level_reference == "current":
        d["kc_signal_basis"] = d["kc_basis"]
        d["kc_signal_upper"] = d["kc_upper"]
        d["kc_signal_lower"] = d["kc_lower"]

    elif cfg.level_reference == "previous":
        d["kc_signal_basis"] = d[
            "kc_basis"
        ].shift(1)

        d["kc_signal_upper"] = d[
            "kc_upper"
        ].shift(1)

        d["kc_signal_lower"] = d[
            "kc_lower"
        ].shift(1)

    else:
        raise ValueError(
            "level_reference must be "
            "'current' or 'previous'."
        )

    d["kc_indicator_ready"] = (
        d["kc_source"].notna()
        & d["kc_atr"].notna()
        & d["kc_basis"].notna()
        & d["kc_signal_upper"].notna()
        & d["kc_signal_lower"].notna()
        & d["kc_signal_basis"].notna()
    )

    if "available_time" in d.columns:
        d["signal_time"] = pd.to_datetime(
            d["available_time"],
            utc=True,
        )
    elif "decision_time" in d.columns:
        d["signal_time"] = pd.to_datetime(
            d["decision_time"],
            utc=True,
        )
    else:
        d["signal_time"] = pd.to_datetime(
            d.index,
            utc=True,
        )

    return d

In [55]:
def crossed_above(
    value: pd.Series,
    level: pd.Series,
) -> pd.Series:
    previous_value = value.shift(1)
    previous_level = level.shift(1)

    ready = (
        value.notna()
        & level.notna()
        & previous_value.notna()
        & previous_level.notna()
    )

    return (
        ready
        & value.gt(level)
        & previous_value.le(previous_level)
    )


def crossed_below(
    value: pd.Series,
    level: pd.Series,
) -> pd.Series:
    previous_value = value.shift(1)
    previous_level = level.shift(1)

    ready = (
        value.notna()
        & level.notna()
        & previous_value.notna()
        & previous_level.notna()
    )

    return (
        ready
        & value.lt(level)
        & previous_value.ge(previous_level)
    )

In [56]:
def add_keltner_cross_events(
    df: pd.DataFrame,
) -> pd.DataFrame:
    required = {
        "close",
        "kc_signal_upper",
        "kc_signal_basis",
        "kc_signal_lower",
        "kc_indicator_ready",
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            f"Missing Keltner features: {sorted(missing)}"
        )

    d = df.copy()

    price = d["close"].astype(float)

    upper = d["kc_signal_upper"]
    basis = d["kc_signal_basis"]
    lower = d["kc_signal_lower"]

    d["kc_close_above_upper"] = (
        price > upper
    )

    d["kc_close_below_lower"] = (
        price < lower
    )

    d["kc_close_inside_channel"] = (
        (price <= upper)
        & (price >= lower)
    )

    d["kc_cross_above_upper"] = crossed_above(
        price,
        upper,
    )

    d["kc_cross_below_upper"] = crossed_below(
        price,
        upper,
    )

    d["kc_cross_above_basis"] = crossed_above(
        price,
        basis,
    )

    d["kc_cross_below_basis"] = crossed_below(
        price,
        basis,
    )

    d["kc_cross_above_lower"] = crossed_above(
        price,
        lower,
    )

    d["kc_cross_below_lower"] = crossed_below(
        price,
        lower,
    )

    event_columns = [
        "kc_close_above_upper",
        "kc_close_below_lower",
        "kc_close_inside_channel",
        "kc_cross_above_upper",
        "kc_cross_below_upper",
        "kc_cross_above_basis",
        "kc_cross_below_basis",
        "kc_cross_above_lower",
        "kc_cross_below_lower",
    ]

    for col in event_columns:
        d[col] = (
            d[col]
            & d["kc_indicator_ready"]
        ).fillna(False)

    return d

In [57]:
def generate_breakout_signals(
    df: pd.DataFrame,
    cfg: KeltnerSignalConfig,
) -> pd.DataFrame:
    d = df.copy()

    d["long_entry_signal"] = (
        d["kc_cross_above_upper"]
        if cfg.allow_long
        else False
    )

    d["short_entry_signal"] = (
        d["kc_cross_below_lower"]
        if cfg.allow_short
        else False
    )

    if cfg.exit_rule == "basis_cross":
        d["long_exit_signal"] = (
            d["kc_cross_below_basis"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_above_basis"]
        )

    elif cfg.exit_rule == "channel_reentry":
        d["long_exit_signal"] = (
            d["kc_cross_below_upper"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_above_lower"]
        )

    elif cfg.exit_rule == "opposite_band":
        d["long_exit_signal"] = (
            d["kc_cross_below_lower"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_above_upper"]
        )

    else:
        raise ValueError(
            f"Unknown breakout exit rule: "
            f"{cfg.exit_rule}"
        )

    return d

In [58]:
def generate_mean_reversion_signals(
    df: pd.DataFrame,
    cfg: KeltnerSignalConfig,
) -> pd.DataFrame:
    d = df.copy()

    d["long_entry_signal"] = (
        d["kc_cross_below_lower"]
        if cfg.allow_long
        else False
    )

    d["short_entry_signal"] = (
        d["kc_cross_above_upper"]
        if cfg.allow_short
        else False
    )

    if cfg.exit_rule == "basis_cross":
        d["long_exit_signal"] = (
            d["kc_cross_above_basis"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_below_basis"]
        )

    elif cfg.exit_rule == "channel_reentry":
        d["long_exit_signal"] = (
            d["kc_cross_above_lower"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_below_upper"]
        )

    elif cfg.exit_rule == "opposite_band":
        d["long_exit_signal"] = (
            d["kc_cross_above_upper"]
        )

        d["short_exit_signal"] = (
            d["kc_cross_below_lower"]
        )

    else:
        raise ValueError(
            f"Unknown mean-reversion exit rule: "
            f"{cfg.exit_rule}"
        )

    return d

In [59]:
def add_signal_diagnostics(
    df: pd.DataFrame,
) -> pd.DataFrame:
    d = df.copy()

    signal_columns = [
        "long_entry_signal",
        "short_entry_signal",
        "long_exit_signal",
        "short_exit_signal",
    ]

    for col in signal_columns:
        d[col] = (
            d[col]
            .fillna(False)
            .astype(bool)
        )

        d.loc[
            ~d["kc_indicator_ready"],
            col,
        ] = False

    d["any_entry_signal"] = (
        d["long_entry_signal"]
        | d["short_entry_signal"]
    )

    d["any_exit_signal"] = (
        d["long_exit_signal"]
        | d["short_exit_signal"]
    )

    d["any_signal"] = (
        d["any_entry_signal"]
        | d["any_exit_signal"]
    )

    d["long_to_short_reversal_signal"] = (
        d["long_exit_signal"]
        & d["short_entry_signal"]
    )

    d["short_to_long_reversal_signal"] = (
        d["short_exit_signal"]
        & d["long_entry_signal"]
    )

    d["entry_conflict"] = (
        d["long_entry_signal"]
        & d["short_entry_signal"]
    )

    signal_counter = (
        d["any_signal"]
        .astype(int)
        .cumsum()
    )

    d["signal_id"] = pd.Series(
        signal_counter,
        index=d.index,
        dtype="Int64",
    ).where(
        d["any_signal"]
    )

    return d

In [60]:
def build_keltner_signal_engine(
    execution_bars: pd.DataFrame,
    *,
    execution_timeframe: str,
    indicator_cfg: KeltnerIndicatorConfig,
    signal_cfg: KeltnerSignalConfig,
) -> KeltnerSignalResult:
    strategy_id = make_keltner_strategy_id(
        execution_timeframe,
        indicator_cfg,
        signal_cfg,
    )

    d = add_keltner_features(
        execution_bars,
        indicator_cfg,
    )

    d = add_keltner_cross_events(d)

    if signal_cfg.strategy_type == "breakout":
        d = generate_breakout_signals(
            d,
            signal_cfg,
        )

    elif signal_cfg.strategy_type == "mean_reversion":
        d = generate_mean_reversion_signals(
            d,
            signal_cfg,
        )

    else:
        raise ValueError(
            f"Unknown strategy type: "
            f"{signal_cfg.strategy_type}"
        )

    d = add_signal_diagnostics(d)

    return KeltnerSignalResult(
        strategy_id=strategy_id,
        execution_timeframe=execution_timeframe,
        indicator_config=indicator_cfg,
        signal_config=signal_cfg,
        data=d,
    )

In [61]:
def validate_keltner_signal_result(
    result: KeltnerSignalResult,
) -> None:
    d = result.data

    required = {
        "signal_time",
        "kc_basis",
        "kc_upper",
        "kc_lower",
        "kc_indicator_ready",

        "long_entry_signal",
        "short_entry_signal",
        "long_exit_signal",
        "short_exit_signal",

        "any_entry_signal",
        "any_exit_signal",
        "any_signal",
        "signal_id",
    }

    missing = required.difference(d.columns)

    if missing:
        raise AssertionError(
            f"Missing signal columns: {sorted(missing)}"
        )

    if not d.index.is_monotonic_increasing:
        raise AssertionError(
            "Signal index is not monotonic."
        )

    if d.index.duplicated().any():
        raise AssertionError(
            "Signal DataFrame contains duplicate timestamps."
        )

    ready = d["kc_indicator_ready"]

    invalid_band_order = (
        ready
        & (
            (d["kc_lower"] > d["kc_basis"])
            | (d["kc_basis"] > d["kc_upper"])
        )
    )

    if invalid_band_order.any():
        raise AssertionError(
            "Invalid Keltner band ordering."
        )

    signal_columns = [
        "long_entry_signal",
        "short_entry_signal",
        "long_exit_signal",
        "short_exit_signal",
    ]

    signal_while_not_ready = (
        ~ready
        & d[signal_columns].any(axis=1)
    )

    if signal_while_not_ready.any():
        raise AssertionError(
            "Signals occurred before indicator was ready."
        )

    if d["entry_conflict"].any():
        bad_rows = d.loc[
            d["entry_conflict"],
            [
                "close",
                "kc_signal_upper",
                "kc_signal_lower",
                "long_entry_signal",
                "short_entry_signal",
            ],
        ].head(10)

        raise AssertionError(
            "Long and short entries occurred simultaneously.\n"
            f"{bad_rows}"
        )

    expected_any_entry = (
        d["long_entry_signal"]
        | d["short_entry_signal"]
    )

    if not (
        d["any_entry_signal"]
        == expected_any_entry
    ).all():
        raise AssertionError(
            "any_entry_signal is inconsistent."
        )

    expected_any_exit = (
        d["long_exit_signal"]
        | d["short_exit_signal"]
    )

    if not (
        d["any_exit_signal"]
        == expected_any_exit
    ).all():
        raise AssertionError(
            "any_exit_signal is inconsistent."
        )

    signal_id_present = d["signal_id"].notna()

    if not (
        signal_id_present
        == d["any_signal"]
    ).all():
        raise AssertionError(
            "signal_id is inconsistent with any_signal."
        )

    signal_time = pd.to_datetime(
        d["signal_time"],
        utc=True,
    )

    index_time = pd.to_datetime(
        d.index,
        utc=True,
    )

    if (signal_time > index_time).any():
        raise AssertionError(
            "Signal time is after its bar timestamp."
        )

In [62]:
def validate_keltner_prefix_invariance(
    execution_bars: pd.DataFrame,
    indicator_cfg: KeltnerIndicatorConfig,
    *,
    checkpoints: list[int] | None = None,
    atol: float = 1e-12,
) -> None:
    if checkpoints is None:
        n = len(execution_bars)

        checkpoints = sorted(
            set(
                [
                    min(n - 1, 100),
                    min(n - 1, 500),
                    min(n - 1, 1_000),
                    n - 1,
                ]
            )
        )

    full = add_keltner_features(
        execution_bars,
        indicator_cfg,
    )

    feature_columns = [
        "kc_source",
        "kc_true_range",
        "kc_atr",
        "kc_basis",
        "kc_upper",
        "kc_lower",
        "kc_signal_basis",
        "kc_signal_upper",
        "kc_signal_lower",
    ]

    for position in checkpoints:
        if position < 0 or position >= len(execution_bars):
            continue

        partial_input = execution_bars.iloc[
            : position + 1
        ]

        partial = add_keltner_features(
            partial_input,
            indicator_cfg,
        )

        for col in feature_columns:
            full_value = full[col].iloc[position]
            partial_value = partial[col].iloc[-1]

            both_nan = (
                pd.isna(full_value)
                and pd.isna(partial_value)
            )

            if both_nan:
                continue

            if not np.isclose(
                full_value,
                partial_value,
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            ):
                raise AssertionError(
                    "Prefix invariance failed: "
                    f"position={position}, "
                    f"column={col}, "
                    f"full={full_value}, "
                    f"partial={partial_value}"
                )

    print(
        "Keltner prefix-invariance validation passed."
    )

In [63]:
# validate_keltner_prefix_invariance(
#     execution_bars=execution_15m,
#     indicator_cfg=keltner_indicator_cfg,
# )

In [64]:
# strategy_signal_library = {
#     breakout_result.strategy_id: breakout_result,
#     mean_reversion_result.strategy_id: (
#         mean_reversion_result
#     ),
# }

สามารถเพิ่ม parameter ชุดใหม่ได้

In [65]:
# wide_breakout_indicator_cfg = (
#     KeltnerIndicatorConfig(
#         ema_period=30,
#         atr_period=20,
#         atr_multiplier=2.5,
#         price_source="hlc3",
#         atr_smoothing="rma",
#         level_reference="current",
#     )
# )

# wide_breakout_result = (
#     build_keltner_signal_engine(
#         execution_bars=execution_15m,
#         execution_timeframe="15min",
#         indicator_cfg=wide_breakout_indicator_cfg,
#         signal_cfg=breakout_cfg,
#     )
# )

# strategy_signal_library[
#     wide_breakout_result.strategy_id
# ] = wide_breakout_result

In [66]:
# strategy_signal_library.keys()

In [67]:
def summarize_keltner_signals(
    result: KeltnerSignalResult,
) -> pd.Series:
    d = result.data

    return pd.Series(
        {
            "strategy_id": result.strategy_id,
            "execution_timeframe": (
                result.execution_timeframe
            ),
            "n_bars": len(d),
            "indicator_ready_bars": int(
                d["kc_indicator_ready"].sum()
            ),
            "long_entries": int(
                d["long_entry_signal"].sum()
            ),
            "short_entries": int(
                d["short_entry_signal"].sum()
            ),
            "long_exits": int(
                d["long_exit_signal"].sum()
            ),
            "short_exits": int(
                d["short_exit_signal"].sum()
            ),
            "bars_with_any_signal": int(
                d["any_signal"].sum()
            ),
            "long_to_short_reversals": int(
                d[
                    "long_to_short_reversal_signal"
                ].sum()
            ),
            "short_to_long_reversals": int(
                d[
                    "short_to_long_reversal_signal"
                ].sum()
            ),
        }
    )

In [68]:
# display(
#     pd.DataFrame(
#         [
#             summarize_keltner_signals(
#                 breakout_result
#             ),
#             summarize_keltner_signals(
#                 mean_reversion_result
#             ),
#         ]
#     )
# )

In [69]:
# signal_view_columns = [
#     "open",
#     "high",
#     "low",
#     "close",

#     "kc_basis",
#     "kc_upper",
#     "kc_lower",

#     "long_entry_signal",
#     "short_entry_signal",
#     "long_exit_signal",
#     "short_exit_signal",

#     "long_to_short_reversal_signal",
#     "short_to_long_reversal_signal",

#     "signal_id",
#     "signal_time",
# ]

In [70]:
# display(
#     breakout_result.data.loc[
#         breakout_result.data["any_signal"],
#         signal_view_columns,
#     ].head(50)
# )

Unit Test ของ Signal Logic

In [71]:
def test_keltner_signal_logic() -> None:
    index = pd.date_range(
        "2025-01-01",
        periods=10,
        freq="15min",
        tz="UTC",
    )

    close = pd.Series(
        [
            100.0,
            106.0,  # cross upper
            104.0,
            99.0,   # cross below basis
            94.0,   # cross lower
            96.0,
            101.0,  # cross above basis
            106.0,  # cross upper
            100.0,
            94.0,   # cross lower
        ],
        index=index,
    )

    test_df = pd.DataFrame(
        {
            "close": close,
            "kc_signal_upper": 105.0,
            "kc_signal_basis": 100.0,
            "kc_signal_lower": 95.0,
            "kc_indicator_ready": True,
        },
        index=index,
    )

    test_df = add_keltner_cross_events(
        test_df
    )

    breakout_cfg = KeltnerSignalConfig(
        strategy_type="breakout",
        exit_rule="basis_cross",
    )

    breakout = generate_breakout_signals(
        test_df,
        breakout_cfg,
    )

    breakout = add_signal_diagnostics(
        breakout
    )

    assert bool(
        breakout.loc[
            index[1],
            "long_entry_signal",
        ]
    )

    assert bool(
        breakout.loc[
            index[3],
            "long_exit_signal",
        ]
    )

    assert bool(
        breakout.loc[
            index[4],
            "short_entry_signal",
        ]
    )

    assert bool(
        breakout.loc[
            index[6],
            "short_exit_signal",
        ]
    )

    mean_reversion_cfg = KeltnerSignalConfig(
        strategy_type="mean_reversion",
        exit_rule="basis_cross",
    )

    mean_reversion = (
        generate_mean_reversion_signals(
            test_df,
            mean_reversion_cfg,
        )
    )

    mean_reversion = add_signal_diagnostics(
        mean_reversion
    )

    assert bool(
        mean_reversion.loc[
            index[1],
            "short_entry_signal",
        ]
    )

    assert bool(
        mean_reversion.loc[
            index[3],
            "short_exit_signal",
        ]
    )

    assert bool(
        mean_reversion.loc[
            index[4],
            "long_entry_signal",
        ]
    )

    assert bool(
        mean_reversion.loc[
            index[6],
            "long_exit_signal",
        ]
    )

    print(
        "Keltner signal logic test passed."
    )

In [72]:
# test_keltner_signal_logic()

# Regime Filter Engine

In [73]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Literal
import hashlib
import json

import numpy as np
import pandas as pd

In [74]:
@dataclass(frozen=True)
class RegimeRule:
    state_family: str
    regime_timeframe: str

    # ถ้าไม่กำหนด หมายถึงไม่จำกัด whitelist
    allowed_states: tuple[str, ...] = ()

    # State เหล่านี้จะถูกปฏิเสธ
    blocked_states: tuple[str, ...] = ()

    # หาก State ไม่พร้อมหรือ alignment invalid
    missing_policy: Literal[
        "reject",
        "allow",
    ] = "reject"

In [75]:
@dataclass(frozen=True)
class RegimeFilterConfig:
    filter_name: str

    enabled: bool = True

    long_rules: tuple[RegimeRule, ...] = ()
    short_rules: tuple[RegimeRule, ...] = ()

    long_combine: Literal["all", "any"] = "all"
    short_combine: Literal["all", "any"] = "all"

    # Filter จะตรวจ State ณ signal_time
    decision_timing: Literal["signal_time"] = "signal_time"

In [76]:
@dataclass
class RegimeFilteredSignalResult:
    source_strategy_id: str
    filter_id: str
    filtered_strategy_id: str

    filter_config: RegimeFilterConfig
    data: pd.DataFrame

In [77]:
def validate_regime_rule(
    rule: RegimeRule,
) -> None:
    if not rule.state_family:
        raise ValueError(
            "state_family cannot be empty."
        )

    if not rule.regime_timeframe:
        raise ValueError(
            "regime_timeframe cannot be empty."
        )

    allowed = set(rule.allowed_states)
    blocked = set(rule.blocked_states)

    overlap = allowed.intersection(blocked)

    if overlap:
        raise ValueError(
            "The same State cannot be both allowed "
            f"and blocked: {sorted(overlap)}"
        )

    if rule.missing_policy not in {
        "reject",
        "allow",
    }:
        raise ValueError(
            f"Unknown missing_policy: "
            f"{rule.missing_policy}"
        )


def validate_regime_filter_config(
    cfg: RegimeFilterConfig,
) -> None:
    if cfg.long_combine not in {"all", "any"}:
        raise ValueError(
            "long_combine must be 'all' or 'any'."
        )

    if cfg.short_combine not in {"all", "any"}:
        raise ValueError(
            "short_combine must be 'all' or 'any'."
        )

    for rule in cfg.long_rules:
        validate_regime_rule(rule)

    for rule in cfg.short_rules:
        validate_regime_rule(rule)

In [78]:
def make_regime_filter_id(
    cfg: RegimeFilterConfig,
) -> str:
    payload = json.dumps(
        asdict(cfg),
        sort_keys=True,
        separators=(",", ":"),
    )

    config_hash = hashlib.sha1(
        payload.encode("utf-8")
    ).hexdigest()[:10]

    filter_name = normalize_column_token(
        cfg.filter_name
    )

    return f"{filter_name}_{config_hash}"

In [79]:
def evaluate_regime_rule(
    df: pd.DataFrame,
    rule: RegimeRule,
) -> pd.DataFrame:
    prefix = make_state_prefix(
        rule.state_family,
        rule.regime_timeframe,
    )

    state_col = (
        f"{prefix}_confirmed_state"
    )

    ready_col = (
        f"{prefix}_state_ready_for_decision"
    )

    alignment_col = (
        f"{prefix}_alignment_valid"
    )

    required = {
        state_col,
        ready_col,
        alignment_col,
    }

    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            "Execution dataset does not contain "
            "the required Regime columns: "
            f"{sorted(missing)}"
        )

    state = df[state_col].astype("string")

    state_ready = (
        df[ready_col].fillna(False).astype(bool)
        & df[alignment_col].fillna(False).astype(bool)
        & state.notna()
    )

    rule_pass = pd.Series(
        True,
        index=df.index,
        dtype=bool,
    )

    reason = pd.Series(
        "passed",
        index=df.index,
        dtype="string",
    )

    missing_state = ~state_ready

    if rule.missing_policy == "reject":
        rule_pass.loc[missing_state] = False
        reason.loc[missing_state] = (
            "state_not_ready"
        )

    elif rule.missing_policy == "allow":
        rule_pass.loc[missing_state] = True
        reason.loc[missing_state] = (
            "missing_state_allowed"
        )

    ready_rows = state_ready

    if rule.allowed_states:
        allowed_mask = state.isin(
            rule.allowed_states
        )

        failed_allowed = (
            ready_rows
            & ~allowed_mask
        )

        rule_pass.loc[failed_allowed] = False
        reason.loc[failed_allowed] = (
            "state_not_allowed"
        )

    if rule.blocked_states:
        blocked_mask = state.isin(
            rule.blocked_states
        )

        failed_blocked = (
            ready_rows
            & blocked_mask
        )

        rule_pass.loc[failed_blocked] = False
        reason.loc[failed_blocked] = (
            "state_blocked"
        )

    return pd.DataFrame(
        {
            "state_family": (
                rule.state_family
            ),
            "regime_timeframe": (
                rule.regime_timeframe
            ),
            "observed_state": state,
            "state_ready": state_ready,
            "rule_pass": rule_pass,
            "rule_reason": reason,
        },
        index=df.index,
    )

รวมหลาย Rules

In [80]:
def combine_rule_passes(
    rule_passes: list[pd.Series],
    *,
    combine_mode: Literal["all", "any"],
    index: pd.Index,
) -> pd.Series:
    if len(rule_passes) == 0:
        return pd.Series(
            True,
            index=index,
            dtype=bool,
        )

    matrix = pd.concat(
        rule_passes,
        axis=1,
    ).fillna(False)

    if combine_mode == "all":
        return matrix.all(axis=1)

    if combine_mode == "any":
        return matrix.any(axis=1)

    raise ValueError(
        f"Unknown combine mode: {combine_mode}"
    )

In [81]:
def evaluate_side_regime_rules(
    df: pd.DataFrame,
    *,
    side: Literal["long", "short"],
    rules: tuple[RegimeRule, ...],
    combine_mode: Literal["all", "any"],
) -> tuple[pd.DataFrame, pd.Series]:
    diagnostics = pd.DataFrame(
        index=df.index
    )

    rule_passes: list[pd.Series] = []

    for rule_number, rule in enumerate(rules):
        evaluation = evaluate_regime_rule(
            df,
            rule,
        )

        rule_prefix = (
            f"rf_{side}_rule{rule_number}"
        )

        diagnostics[
            f"{rule_prefix}_family"
        ] = rule.state_family

        diagnostics[
            f"{rule_prefix}_timeframe"
        ] = rule.regime_timeframe

        diagnostics[
            f"{rule_prefix}_observed_state"
        ] = evaluation["observed_state"]

        diagnostics[
            f"{rule_prefix}_state_ready"
        ] = evaluation["state_ready"]

        diagnostics[
            f"{rule_prefix}_pass"
        ] = evaluation["rule_pass"]

        diagnostics[
            f"{rule_prefix}_reason"
        ] = evaluation["rule_reason"]

        rule_passes.append(
            evaluation["rule_pass"]
        )

    side_pass = combine_rule_passes(
        rule_passes,
        combine_mode=combine_mode,
        index=df.index,
    )

    diagnostics[
        f"{side}_regime_pass"
    ] = side_pass

    return diagnostics, side_pass

In [82]:
def build_rejection_reason(
    diagnostics: pd.DataFrame,
    *,
    side: Literal["long", "short"],
    raw_entry_signal: pd.Series,
    regime_pass: pd.Series,
) -> pd.Series:
    result = pd.Series(
        pd.NA,
        index=diagnostics.index,
        dtype="string",
    )

    rejected = (
        raw_entry_signal
        & ~regime_pass
    )

    reason_columns = [
        col
        for col in diagnostics.columns
        if (
            col.startswith(f"rf_{side}_rule")
            and col.endswith("_reason")
        )
    ]

    if not reason_columns:
        result.loc[rejected] = (
            "filter_disabled_or_no_rules"
        )
        return result

    for timestamp in diagnostics.index[rejected]:
        failed_reasons = []

        for reason_col in reason_columns:
            reason = diagnostics.at[
                timestamp,
                reason_col,
            ]

            if pd.isna(reason):
                continue

            if reason in {
                "passed",
                "missing_state_allowed",
            }:
                continue

            rule_prefix = reason_col.removesuffix(
                "_reason"
            )

            timeframe_col = (
                f"{rule_prefix}_timeframe"
            )

            state_col = (
                f"{rule_prefix}_observed_state"
            )

            timeframe = diagnostics.at[
                timestamp,
                timeframe_col,
            ]

            observed_state = diagnostics.at[
                timestamp,
                state_col,
            ]

            failed_reasons.append(
                f"{timeframe}:"
                f"{reason}:"
                f"{observed_state}"
            )

        if failed_reasons:
            result.at[timestamp] = ";".join(
                failed_reasons
            )
        else:
            result.at[timestamp] = (
                "combined_rule_failed"
            )

    return result

In [83]:
def apply_regime_filter(
    signal_result: KeltnerSignalResult,
    filter_cfg: RegimeFilterConfig,
) -> RegimeFilteredSignalResult:
    validate_regime_filter_config(
        filter_cfg
    )

    d = signal_result.data.copy()

    required_signals = {
        "signal_time",
        "long_entry_signal",
        "short_entry_signal",
        "long_exit_signal",
        "short_exit_signal",
    }

    missing = required_signals.difference(
        d.columns
    )

    if missing:
        raise ValueError(
            f"Missing raw signal columns: "
            f"{sorted(missing)}"
        )

    d["raw_long_entry_signal"] = (
        d["long_entry_signal"]
        .fillna(False)
        .astype(bool)
    )

    d["raw_short_entry_signal"] = (
        d["short_entry_signal"]
        .fillna(False)
        .astype(bool)
    )

    d["raw_long_exit_signal"] = (
        d["long_exit_signal"]
        .fillna(False)
        .astype(bool)
    )

    d["raw_short_exit_signal"] = (
        d["short_exit_signal"]
        .fillna(False)
        .astype(bool)
    )

    if filter_cfg.enabled:
        long_diagnostics, long_pass = (
            evaluate_side_regime_rules(
                d,
                side="long",
                rules=filter_cfg.long_rules,
                combine_mode=(
                    filter_cfg.long_combine
                ),
            )
        )

        short_diagnostics, short_pass = (
            evaluate_side_regime_rules(
                d,
                side="short",
                rules=filter_cfg.short_rules,
                combine_mode=(
                    filter_cfg.short_combine
                ),
            )
        )

        d = d.join(long_diagnostics)
        d = d.join(short_diagnostics)

    else:
        long_pass = pd.Series(
            True,
            index=d.index,
            dtype=bool,
        )

        short_pass = pd.Series(
            True,
            index=d.index,
            dtype=bool,
        )

        d["long_regime_pass"] = True
        d["short_regime_pass"] = True

    d["filtered_long_entry_signal"] = (
        d["raw_long_entry_signal"]
        & long_pass
    )

    d["filtered_short_entry_signal"] = (
        d["raw_short_entry_signal"]
        & short_pass
    )

    # Exit ไม่ถูก Filter
    d["filtered_long_exit_signal"] = (
        d["raw_long_exit_signal"]
    )

    d["filtered_short_exit_signal"] = (
        d["raw_short_exit_signal"]
    )

    d["long_entry_rejected"] = (
        d["raw_long_entry_signal"]
        & ~d["filtered_long_entry_signal"]
    )

    d["short_entry_rejected"] = (
        d["raw_short_entry_signal"]
        & ~d["filtered_short_entry_signal"]
    )

    if filter_cfg.enabled:
        d["long_rejection_reason"] = (
            build_rejection_reason(
                d,
                side="long",
                raw_entry_signal=(
                    d["raw_long_entry_signal"]
                ),
                regime_pass=long_pass,
            )
        )

        d["short_rejection_reason"] = (
            build_rejection_reason(
                d,
                side="short",
                raw_entry_signal=(
                    d["raw_short_entry_signal"]
                ),
                regime_pass=short_pass,
            )
        )

    else:
        d["long_rejection_reason"] = (
            pd.Series(
                pd.NA,
                index=d.index,
                dtype="string",
            )
        )

        d["short_rejection_reason"] = (
            pd.Series(
                pd.NA,
                index=d.index,
                dtype="string",
            )
        )

    d["filtered_any_entry_signal"] = (
        d["filtered_long_entry_signal"]
        | d["filtered_short_entry_signal"]
    )

    d["filtered_any_exit_signal"] = (
        d["filtered_long_exit_signal"]
        | d["filtered_short_exit_signal"]
    )

    d["filter_decision_time"] = pd.to_datetime(
        d["signal_time"],
        utc=True,
    )

    filter_id = make_regime_filter_id(
        filter_cfg
    )

    filtered_strategy_id = (
        f"{signal_result.strategy_id}"
        f"__rf_{filter_id}"
    )

    d["regime_filter_id"] = filter_id
    d["filtered_strategy_id"] = (
        filtered_strategy_id
    )

    return RegimeFilteredSignalResult(
        source_strategy_id=(
            signal_result.strategy_id
        ),
        filter_id=filter_id,
        filtered_strategy_id=(
            filtered_strategy_id
        ),
        filter_config=filter_cfg,
        data=d,
    )

In [84]:
def validate_filtered_signal_result(
    result: RegimeFilteredSignalResult,
) -> None:
    d = result.data

    required = {
        "raw_long_entry_signal",
        "raw_short_entry_signal",
        "raw_long_exit_signal",
        "raw_short_exit_signal",

        "filtered_long_entry_signal",
        "filtered_short_entry_signal",
        "filtered_long_exit_signal",
        "filtered_short_exit_signal",

        "long_entry_rejected",
        "short_entry_rejected",

        "filter_decision_time",
    }

    missing = required.difference(d.columns)

    if missing:
        raise AssertionError(
            f"Missing filtered signal columns: "
            f"{sorted(missing)}"
        )

    filtered_without_raw_long = (
        d["filtered_long_entry_signal"]
        & ~d["raw_long_entry_signal"]
    )

    if filtered_without_raw_long.any():
        raise AssertionError(
            "Filter created a new long entry signal."
        )

    filtered_without_raw_short = (
        d["filtered_short_entry_signal"]
        & ~d["raw_short_entry_signal"]
    )

    if filtered_without_raw_short.any():
        raise AssertionError(
            "Filter created a new short entry signal."
        )

    if not (
        d["filtered_long_exit_signal"]
        == d["raw_long_exit_signal"]
    ).all():
        raise AssertionError(
            "Regime filter modified long exits."
        )

    if not (
        d["filtered_short_exit_signal"]
        == d["raw_short_exit_signal"]
    ).all():
        raise AssertionError(
            "Regime filter modified short exits."
        )

    expected_long_rejected = (
        d["raw_long_entry_signal"]
        & ~d["filtered_long_entry_signal"]
    )

    if not (
        d["long_entry_rejected"]
        == expected_long_rejected
    ).all():
        raise AssertionError(
            "long_entry_rejected is inconsistent."
        )

    expected_short_rejected = (
        d["raw_short_entry_signal"]
        & ~d["filtered_short_entry_signal"]
    )

    if not (
        d["short_entry_rejected"]
        == expected_short_rejected
    ).all():
        raise AssertionError(
            "short_entry_rejected is inconsistent."
        )

    rejection_without_reason = (
        d["long_entry_rejected"]
        & d["long_rejection_reason"].isna()
    )

    if rejection_without_reason.any():
        raise AssertionError(
            "Rejected long signal has no reason."
        )

    rejection_without_reason = (
        d["short_entry_rejected"]
        & d["short_rejection_reason"].isna()
    )

    if rejection_without_reason.any():
        raise AssertionError(
            "Rejected short signal has no reason."
        )

    signal_time = pd.to_datetime(
        d["signal_time"],
        utc=True,
    )

    filter_time = pd.to_datetime(
        d["filter_decision_time"],
        utc=True,
    )

    if not signal_time.eq(filter_time).all():
        raise AssertionError(
            "Filter was not evaluated at signal_time."
        )

Baseline แบบไม่ใช้ Regime

ตัวอย่าง Directional Breakout Filter

ตัวอย่าง Mean-Reversion Flat Filter

ตัวอย่าง Multi-Timeframe Filter

In [85]:
def summarize_regime_filter(
    result: RegimeFilteredSignalResult,
) -> pd.Series:
    d = result.data

    raw_long = int(
        d["raw_long_entry_signal"].sum()
    )

    raw_short = int(
        d["raw_short_entry_signal"].sum()
    )

    accepted_long = int(
        d["filtered_long_entry_signal"].sum()
    )

    accepted_short = int(
        d["filtered_short_entry_signal"].sum()
    )

    rejected_long = int(
        d["long_entry_rejected"].sum()
    )

    rejected_short = int(
        d["short_entry_rejected"].sum()
    )

    return pd.Series(
        {
            "source_strategy_id": (
                result.source_strategy_id
            ),
            "filter_id": result.filter_id,

            "raw_long_entries": raw_long,
            "accepted_long_entries": (
                accepted_long
            ),
            "rejected_long_entries": (
                rejected_long
            ),
            "long_acceptance_ratio": (
                accepted_long / raw_long
                if raw_long > 0
                else np.nan
            ),

            "raw_short_entries": raw_short,
            "accepted_short_entries": (
                accepted_short
            ),
            "rejected_short_entries": (
                rejected_short
            ),
            "short_acceptance_ratio": (
                accepted_short / raw_short
                if raw_short > 0
                else np.nan
            ),

            "raw_total_entries": (
                raw_long + raw_short
            ),
            "accepted_total_entries": (
                accepted_long + accepted_short
            ),
            "rejected_total_entries": (
                rejected_long + rejected_short
            ),
        }
    )

In [86]:
def extract_rejected_entry_audit(
    result: RegimeFilteredSignalResult,
) -> pd.DataFrame:
    d = result.data

    rejected = (
        d["long_entry_rejected"]
        | d["short_entry_rejected"]
    )

    base_columns = [
        "signal_time",
        "open",
        "high",
        "low",
        "close",
        "kc_basis",
        "kc_upper",
        "kc_lower",

        "raw_long_entry_signal",
        "raw_short_entry_signal",

        "filtered_long_entry_signal",
        "filtered_short_entry_signal",

        "long_entry_rejected",
        "short_entry_rejected",

        "long_rejection_reason",
        "short_rejection_reason",
    ]

    diagnostic_columns = [
        col
        for col in d.columns
        if col.startswith("rf_")
    ]

    selected_columns = [
        col
        for col in (
            base_columns
            + diagnostic_columns
        )
        if col in d.columns
    ]

    return d.loc[
        rejected,
        selected_columns,
    ].copy()

In [87]:
def extract_accepted_entry_audit(
    result: RegimeFilteredSignalResult,
) -> pd.DataFrame:
    d = result.data

    accepted = (
        d["filtered_long_entry_signal"]
        | d["filtered_short_entry_signal"]
    )

    columns = [
        "signal_time",
        "open",
        "high",
        "low",
        "close",

        "kc_basis",
        "kc_upper",
        "kc_lower",

        "filtered_long_entry_signal",
        "filtered_short_entry_signal",
    ]

    regime_columns = [
        col
        for col in d.columns
        if col.endswith(
            "_confirmed_state"
        )
    ]

    columns = [
        col
        for col in columns + regime_columns
        if col in d.columns
    ]

    return d.loc[
        accepted,
        columns,
    ].copy()

In [88]:
def test_regime_filter_logic() -> None:
    index = pd.date_range(
        "2025-01-01",
        periods=5,
        freq="15min",
        tz="UTC",
    )

    prefix = make_state_prefix(
        "imom_imacd",
        "4h",
    )

    test_data = pd.DataFrame(
        {
            "signal_time": index,

            "long_entry_signal": [
                True,
                True,
                True,
                False,
                False,
            ],

            "short_entry_signal": [
                False,
                False,
                False,
                True,
                True,
            ],

            "long_exit_signal": False,
            "short_exit_signal": False,

            f"{prefix}_confirmed_state": [
                "plus_lime",
                "flat_orange",
                pd.NA,
                "minus_red",
                "plus_lime",
            ],

            f"{prefix}_state_ready_for_decision": [
                True,
                True,
                False,
                True,
                True,
            ],

            f"{prefix}_alignment_valid": [
                True,
                True,
                False,
                True,
                True,
            ],
        },
        index=index,
    )

    dummy_result = KeltnerSignalResult(
        strategy_id="test_keltner",
        execution_timeframe="15min",
        indicator_config=(
            KeltnerIndicatorConfig()
        ),
        signal_config=KeltnerSignalConfig(
            strategy_type="breakout"
        ),
        data=test_data,
    )

    cfg = RegimeFilterConfig(
        filter_name="test_directional",

        long_rules=(
            RegimeRule(
                state_family="imom_imacd",
                regime_timeframe="4h",
                allowed_states=(
                    "plus_lime",
                ),
            ),
        ),

        short_rules=(
            RegimeRule(
                state_family="imom_imacd",
                regime_timeframe="4h",
                allowed_states=(
                    "minus_red",
                ),
            ),
        ),
    )

    result = apply_regime_filter(
        dummy_result,
        cfg,
    )

    d = result.data

    # plus_lime long ผ่าน
    assert bool(
        d.loc[
            index[0],
            "filtered_long_entry_signal",
        ]
    )

    # flat_orange long ไม่ผ่าน
    assert not bool(
        d.loc[
            index[1],
            "filtered_long_entry_signal",
        ]
    )

    # State ไม่พร้อม ไม่ผ่าน
    assert not bool(
        d.loc[
            index[2],
            "filtered_long_entry_signal",
        ]
    )

    # minus_red short ผ่าน
    assert bool(
        d.loc[
            index[3],
            "filtered_short_entry_signal",
        ]
    )

    # plus_lime short ไม่ผ่าน
    assert not bool(
        d.loc[
            index[4],
            "filtered_short_entry_signal",
        ]
    )

    validate_filtered_signal_result(
        result
    )

    print(
        "Regime filter logic test passed."
    )

In [89]:
# test_regime_filter_logic()

In [90]:
# 1-minute OHLC
#     ↓
# Time Bars
#     ↓
# IMOM × IMACD State
#     ↓
# Causal Persistence
#     ↓
# Point-in-Time Alignment
#     ↓
# Raw Keltner Signal
#     ↓
# Regime Filter Decision
#     ↓
# Accepted / Rejected Entry Signals

# Execution Simulator

## Single Position

In [91]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal, Optional

import numpy as np
import pandas as pd

In [92]:
@dataclass(frozen=True)
class ExecutionConfig:
    initial_capital: float = 100_000.0

    # คิดเป็น basis points
    # 1 bps = 0.01%
    fee_bps: float = 4.0
    slippage_bps: float = 1.0

    position_sizing: Literal[
        "fraction_of_equity",
        "fixed_notional",
    ] = "fraction_of_equity"

    # ใช้เมื่อ position_sizing="fraction_of_equity"
    position_fraction: float = 1.0
    leverage: float = 1.0

    # ใช้เมื่อ position_sizing="fixed_notional"
    fixed_notional: float = 10_000.0

    min_notional: float = 0.0

    # 1 = signal จากแท่ง t ไป fill ที่ open ของแท่งถัดไป
    execution_delay_bars: int = 1

    allow_reversal: bool = True
    allow_same_bar_reentry: bool = False

    entry_conflict_policy: Literal[
        "skip",
    ] = "skip"

    # ปิด Position ที่ close ของแท่งสุดท้ายหรือไม่
    # False เหมาะกับการดูข้อมูลต่อเนื่อง
    # True เหมาะกับ Fold ที่ต้องการปิดบัญชีแยกกัน
    close_position_at_end: bool = False

In [93]:
@dataclass
class OpenPosition:
    trade_id: int

    side: Literal["long", "short"]
    sign: int

    quantity: float

    entry_signal_time: pd.Timestamp
    entry_signal_bar_end: pd.Timestamp

    entry_fill_time: pd.Timestamp
    entry_fill_bar_end: pd.Timestamp
    entry_bar_position: int

    entry_raw_price: float
    entry_fill_price: float

    entry_notional: float
    entry_fee: float
    entry_slippage_cost: float

    mfe_pnl: float = 0.0
    mae_pnl: float = 0.0

    mfe_time: Optional[pd.Timestamp] = None
    mae_time: Optional[pd.Timestamp] = None

    entry_context: dict[str, object] = field(
        default_factory=dict
    )

In [94]:
@dataclass
class ExecutionResult:
    strategy_id: str
    execution_timeframe: str
    config: ExecutionConfig

    bars: pd.DataFrame
    decisions: pd.DataFrame
    fills: pd.DataFrame
    trades: pd.DataFrame

    open_position: Optional[dict[str, object]]

In [95]:
def validate_execution_config(
    cfg: ExecutionConfig,
) -> None:
    if cfg.initial_capital <= 0:
        raise ValueError(
            "initial_capital must be positive."
        )

    if cfg.fee_bps < 0:
        raise ValueError(
            "fee_bps cannot be negative."
        )

    if cfg.slippage_bps < 0:
        raise ValueError(
            "slippage_bps cannot be negative."
        )

    if cfg.execution_delay_bars < 1:
        raise ValueError(
            "execution_delay_bars must be at least 1."
        )

    if cfg.position_fraction <= 0:
        raise ValueError(
            "position_fraction must be positive."
        )

    if cfg.leverage <= 0:
        raise ValueError(
            "leverage must be positive."
        )

    if cfg.fixed_notional <= 0:
        raise ValueError(
            "fixed_notional must be positive."
        )

    if cfg.min_notional < 0:
        raise ValueError(
            "min_notional cannot be negative."
        )

In [96]:
def apply_execution_slippage(
    raw_price: float,
    *,
    transaction_side: Literal["buy", "sell"],
    slippage_bps: float,
) -> float:
    slippage_rate = slippage_bps / 10_000.0

    if transaction_side == "buy":
        return raw_price * (1.0 + slippage_rate)

    if transaction_side == "sell":
        return raw_price * (1.0 - slippage_rate)

    raise ValueError(
        f"Unknown transaction side: {transaction_side}"
    )

In [97]:
def calculate_target_notional(
    realized_equity: float,
    cfg: ExecutionConfig,
) -> float:
    if cfg.position_sizing == "fraction_of_equity":
        usable_equity = max(
            float(realized_equity),
            0.0,
        )

        target_notional = (
            usable_equity
            * cfg.position_fraction
            * cfg.leverage
        )

    elif cfg.position_sizing == "fixed_notional":
        target_notional = cfg.fixed_notional

    else:
        raise ValueError(
            f"Unknown position sizing: "
            f"{cfg.position_sizing}"
        )

    if target_notional < cfg.min_notional:
        return 0.0

    return float(target_notional)

In [98]:
def infer_trade_context_columns(
    df: pd.DataFrame,
) -> list[str]:
    columns: list[str] = []

    base_candidates = [
        "close",
        "kc_basis",
        "kc_upper",
        "kc_lower",
        "kc_atr",
        "imom_score",
    ]

    for col in base_candidates:
        if col in df.columns:
            columns.append(col)

    for col in df.columns:
        if col.endswith("_confirmed_state"):
            columns.append(col)

        elif col.endswith("_state_elapsed_time"):
            columns.append(col)

    return list(dict.fromkeys(columns))

In [99]:
def to_python_scalar(
    value: object,
) -> object:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, np.generic):
        return value.item()

    return value

In [100]:
def capture_signal_context(
    row: pd.Series,
    context_columns: list[str],
    *,
    prefix: str,
) -> dict[str, object]:
    result: dict[str, object] = {}

    for col in context_columns:
        if col not in row.index:
            continue

        result[f"{prefix}{col}"] = (
            to_python_scalar(row[col])
        )

    return result

In [101]:
def update_position_excursions(
    position: OpenPosition,
    *,
    bar_high: float,
    bar_low: float,
    bar_end: pd.Timestamp,
) -> None:
    if position.side == "long":
        favorable_pnl = (
            position.quantity
            * (
                bar_high
                - position.entry_fill_price
            )
        )

        adverse_pnl = (
            position.quantity
            * (
                bar_low
                - position.entry_fill_price
            )
        )

    else:
        favorable_pnl = (
            position.quantity
            * (
                position.entry_fill_price
                - bar_low
            )
        )

        adverse_pnl = (
            position.quantity
            * (
                position.entry_fill_price
                - bar_high
            )
        )

    if favorable_pnl > position.mfe_pnl:
        position.mfe_pnl = favorable_pnl
        position.mfe_time = bar_end

    if adverse_pnl < position.mae_pnl:
        position.mae_pnl = adverse_pnl
        position.mae_time = bar_end

In [102]:
def run_execution_simulator(
    filtered_result: RegimeFilteredSignalResult,
    *,
    execution_timeframe: str,
    cfg: ExecutionConfig,
    context_columns: Optional[list[str]] = None,
) -> ExecutionResult:
    validate_execution_config(cfg)

    d = filtered_result.data.copy()

    required = {
        "open",
        "high",
        "low",
        "close",
        "signal_time",

        "filtered_long_entry_signal",
        "filtered_short_entry_signal",
        "filtered_long_exit_signal",
        "filtered_short_exit_signal",
    }

    missing = required.difference(d.columns)

    if missing:
        raise ValueError(
            f"Missing execution columns: "
            f"{sorted(missing)}"
        )

    if not isinstance(
        d.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "Execution data index must be DatetimeIndex."
        )

    if not d.index.is_monotonic_increasing:
        raise ValueError(
            "Execution data must be sorted by time."
        )

    if d.index.duplicated().any():
        raise ValueError(
            "Execution data contains duplicated timestamps."
        )

    if context_columns is None:
        context_columns = (
            infer_trade_context_columns(d)
        )

    bar_duration = pd.Timedelta(
        execution_timeframe
    )

    if "bar_start" not in d.columns:
        d["bar_start"] = (
            d.index - bar_duration
        )

    d["bar_start"] = pd.to_datetime(
        d["bar_start"],
        utc=True,
    )

    d["signal_time"] = pd.to_datetime(
        d["signal_time"],
        utc=True,
    )

    signal_columns = [
        "filtered_long_entry_signal",
        "filtered_short_entry_signal",
        "filtered_long_exit_signal",
        "filtered_short_exit_signal",
    ]

    for col in signal_columns:
        d[col] = (
            d[col]
            .fillna(False)
            .astype(bool)
        )

    n = len(d)

    cash = float(cfg.initial_capital)
    position: Optional[OpenPosition] = None

    next_trade_id = 1
    next_fill_id = 1
    next_decision_id = 1

    decision_rows: list[dict] = []
    fill_rows: list[dict] = []
    trade_rows: list[dict] = []

    cash_series = np.full(
        n,
        np.nan,
        dtype=float,
    )

    realized_equity_series = np.full(
        n,
        np.nan,
        dtype=float,
    )

    unrealized_pnl_series = np.full(
        n,
        np.nan,
        dtype=float,
    )

    mtm_equity_series = np.full(
        n,
        np.nan,
        dtype=float,
    )

    position_sign_series = np.zeros(
        n,
        dtype=np.int8,
    )

    position_qty_series = np.zeros(
        n,
        dtype=float,
    )

    position_notional_series = np.zeros(
        n,
        dtype=float,
    )

    trade_id_series = np.full(
        n,
        -1,
        dtype=np.int64,
    )

    fill_count_series = np.zeros(
        n,
        dtype=np.int64,
    )

    executed_entry_series = np.zeros(
        n,
        dtype=bool,
    )

    executed_exit_series = np.zeros(
        n,
        dtype=bool,
    )

    executed_reversal_series = np.zeros(
        n,
        dtype=bool,
    )

    fee_rate = cfg.fee_bps / 10_000.0

    def close_current_position(
        *,
        raw_price: float,
        fill_time: pd.Timestamp,
        fill_bar_end: pd.Timestamp,
        fill_bar_position: int,
        exit_signal_time: Optional[pd.Timestamp],
        exit_signal_bar_end: Optional[pd.Timestamp],
        exit_reason: str,
        decision_id: Optional[int],
        exit_context: Optional[
            dict[str, object]
        ] = None,
        exit_at_bar_close: bool = False,
    ) -> str:
        nonlocal cash
        nonlocal position
        nonlocal next_fill_id

        if position is None:
            raise RuntimeError(
                "Cannot close an empty position."
            )

        closed_position = position

        transaction_side: Literal[
            "buy",
            "sell",
        ]

        if closed_position.side == "long":
            transaction_side = "sell"
        else:
            transaction_side = "buy"

        fill_price = apply_execution_slippage(
            raw_price,
            transaction_side=transaction_side,
            slippage_bps=cfg.slippage_bps,
        )

        exit_notional = (
            closed_position.quantity
            * fill_price
        )

        exit_fee = (
            abs(exit_notional)
            * fee_rate
        )

        exit_slippage_cost = (
            abs(fill_price - raw_price)
            * closed_position.quantity
        )

        gross_pnl = (
            closed_position.sign
            * closed_position.quantity
            * (
                fill_price
                - closed_position.entry_fill_price
            )
        )

        market_gross_pnl = (
            closed_position.sign
            * closed_position.quantity
            * (
                raw_price
                - closed_position.entry_raw_price
            )
        )

        total_fees = (
            closed_position.entry_fee
            + exit_fee
        )

        total_slippage_cost = (
            closed_position.entry_slippage_cost
            + exit_slippage_cost
        )

        net_pnl = (
            gross_pnl
            - total_fees
        )

        cash += gross_pnl - exit_fee

        fill_rows.append(
            {
                "fill_id": next_fill_id,
                "decision_id": decision_id,
                "trade_id": (
                    closed_position.trade_id
                ),

                "fill_action": "exit",
                "position_side": (
                    closed_position.side
                ),
                "transaction_side": (
                    transaction_side
                ),

                "signal_time": (
                    exit_signal_time
                ),
                "signal_bar_end": (
                    exit_signal_bar_end
                ),

                "fill_time": fill_time,
                "fill_bar_end": fill_bar_end,

                "raw_price": raw_price,
                "fill_price": fill_price,

                "quantity": (
                    closed_position.quantity
                ),
                "notional": exit_notional,

                "fee": exit_fee,
                "slippage_cost": (
                    exit_slippage_cost
                ),

                "cash_after_fill": cash,
                "exit_reason": exit_reason,
            }
        )

        next_fill_id += 1

        if exit_at_bar_close:
            holding_bars = (
                fill_bar_position
                - closed_position.entry_bar_position
                + 1
            )
        else:
            holding_bars = (
                fill_bar_position
                - closed_position.entry_bar_position
            )

        holding_duration = (
            fill_time
            - closed_position.entry_fill_time
        )

        trade_row = {
            "trade_id": (
                closed_position.trade_id
            ),
            "side": closed_position.side,

            "entry_signal_time": (
                closed_position.entry_signal_time
            ),
            "entry_signal_bar_end": (
                closed_position.entry_signal_bar_end
            ),
            "entry_fill_time": (
                closed_position.entry_fill_time
            ),
            "entry_fill_bar_end": (
                closed_position.entry_fill_bar_end
            ),

            "exit_signal_time": exit_signal_time,
            "exit_signal_bar_end": (
                exit_signal_bar_end
            ),
            "exit_fill_time": fill_time,
            "exit_fill_bar_end": fill_bar_end,

            "entry_raw_price": (
                closed_position.entry_raw_price
            ),
            "entry_fill_price": (
                closed_position.entry_fill_price
            ),

            "exit_raw_price": raw_price,
            "exit_fill_price": fill_price,

            "quantity": (
                closed_position.quantity
            ),
            "entry_notional": (
                closed_position.entry_notional
            ),

            "market_gross_pnl": (
                market_gross_pnl
            ),
            "gross_pnl_after_slippage": (
                gross_pnl
            ),
            "total_slippage_cost": (
                total_slippage_cost
            ),
            "entry_fee": (
                closed_position.entry_fee
            ),
            "exit_fee": exit_fee,
            "total_fees": total_fees,
            "net_pnl": net_pnl,

            "gross_return_on_entry": (
                gross_pnl
                / closed_position.entry_notional
                if closed_position.entry_notional > 0
                else np.nan
            ),
            "net_return_on_entry": (
                net_pnl
                / closed_position.entry_notional
                if closed_position.entry_notional > 0
                else np.nan
            ),

            "holding_bars": holding_bars,
            "holding_duration": (
                holding_duration
            ),

            "mfe_pnl": (
                closed_position.mfe_pnl
            ),
            "mae_pnl": (
                closed_position.mae_pnl
            ),

            "mfe_return_on_entry": (
                closed_position.mfe_pnl
                / closed_position.entry_notional
                if closed_position.entry_notional > 0
                else np.nan
            ),

            "mae_return_on_entry": (
                closed_position.mae_pnl
                / closed_position.entry_notional
                if closed_position.entry_notional > 0
                else np.nan
            ),

            "mfe_time": (
                closed_position.mfe_time
            ),
            "mae_time": (
                closed_position.mae_time
            ),

            "exit_reason": exit_reason,
        }

        trade_row.update(
            closed_position.entry_context
        )

        if exit_context:
            trade_row.update(exit_context)

        trade_rows.append(trade_row)

        closed_side = closed_position.side
        position = None

        return closed_side

    def open_new_position(
        *,
        side: Literal["long", "short"],
        raw_price: float,
        fill_time: pd.Timestamp,
        fill_bar_end: pd.Timestamp,
        fill_bar_position: int,
        entry_signal_time: pd.Timestamp,
        entry_signal_bar_end: pd.Timestamp,
        decision_id: int,
        entry_context: dict[str, object],
    ) -> tuple[bool, str]:
        nonlocal cash
        nonlocal position
        nonlocal next_trade_id
        nonlocal next_fill_id

        if position is not None:
            return False, "position_already_open"

        target_notional = (
            calculate_target_notional(
                cash,
                cfg,
            )
        )

        if target_notional <= 0:
            return False, "insufficient_equity"

        if side == "long":
            transaction_side: Literal[
                "buy",
                "sell",
            ] = "buy"
            sign = 1

        else:
            transaction_side = "sell"
            sign = -1

        fill_price = apply_execution_slippage(
            raw_price,
            transaction_side=transaction_side,
            slippage_bps=cfg.slippage_bps,
        )

        quantity = (
            target_notional
            / fill_price
        )

        actual_notional = (
            quantity
            * fill_price
        )

        entry_fee = (
            abs(actual_notional)
            * fee_rate
        )

        entry_slippage_cost = (
            abs(fill_price - raw_price)
            * quantity
        )

        cash -= entry_fee

        position = OpenPosition(
            trade_id=next_trade_id,

            side=side,
            sign=sign,
            quantity=quantity,

            entry_signal_time=(
                entry_signal_time
            ),
            entry_signal_bar_end=(
                entry_signal_bar_end
            ),

            entry_fill_time=fill_time,
            entry_fill_bar_end=fill_bar_end,
            entry_bar_position=(
                fill_bar_position
            ),

            entry_raw_price=raw_price,
            entry_fill_price=fill_price,

            entry_notional=actual_notional,
            entry_fee=entry_fee,
            entry_slippage_cost=(
                entry_slippage_cost
            ),

            entry_context=entry_context,
        )

        fill_rows.append(
            {
                "fill_id": next_fill_id,
                "decision_id": decision_id,
                "trade_id": next_trade_id,

                "fill_action": "entry",
                "position_side": side,
                "transaction_side": (
                    transaction_side
                ),

                "signal_time": (
                    entry_signal_time
                ),
                "signal_bar_end": (
                    entry_signal_bar_end
                ),

                "fill_time": fill_time,
                "fill_bar_end": fill_bar_end,

                "raw_price": raw_price,
                "fill_price": fill_price,

                "quantity": quantity,
                "notional": actual_notional,

                "fee": entry_fee,
                "slippage_cost": (
                    entry_slippage_cost
                ),

                "cash_after_fill": cash,
                "exit_reason": None,
            }
        )

        next_fill_id += 1
        next_trade_id += 1

        return True, f"opened_{side}"

    for bar_position in range(n):
        bar_end = pd.Timestamp(
            d.index[bar_position]
        )

        row = d.iloc[bar_position]

        bar_start = pd.Timestamp(
            row["bar_start"]
        )

        raw_open = float(row["open"])
        bar_high = float(row["high"])
        bar_low = float(row["low"])
        bar_close = float(row["close"])

        source_position = (
            bar_position
            - cfg.execution_delay_bars
        )

        if source_position >= 0:
            signal_row = d.iloc[
                source_position
            ]

            signal_bar_end = pd.Timestamp(
                d.index[source_position]
            )

            signal_time = pd.Timestamp(
                signal_row["signal_time"]
            )

            long_entry = bool(
                signal_row[
                    "filtered_long_entry_signal"
                ]
            )

            short_entry = bool(
                signal_row[
                    "filtered_short_entry_signal"
                ]
            )

            long_exit = bool(
                signal_row[
                    "filtered_long_exit_signal"
                ]
            )

            short_exit = bool(
                signal_row[
                    "filtered_short_exit_signal"
                ]
            )

            has_signal = (
                long_entry
                or short_entry
                or long_exit
                or short_exit
            )

            if has_signal:
                decision_id = next_decision_id
                next_decision_id += 1

                position_before = (
                    position.side
                    if position is not None
                    else "flat"
                )

                actions: list[str] = []
                closed_side: Optional[str] = None
                opened_side: Optional[str] = None

                entry_context = (
                    capture_signal_context(
                        signal_row,
                        context_columns,
                        prefix="entry_signal__",
                    )
                )

                exit_context = (
                    capture_signal_context(
                        signal_row,
                        context_columns,
                        prefix="exit_signal__",
                    )
                )

                # -------------------------
                # 1. ปิด Position เดิมก่อน
                # -------------------------

                if position is not None:
                    explicit_exit = (
                        (
                            position.side == "long"
                            and long_exit
                        )
                        or (
                            position.side == "short"
                            and short_exit
                        )
                    )

                    opposite_entry = (
                        (
                            position.side == "long"
                            and short_entry
                        )
                        or (
                            position.side == "short"
                            and long_entry
                        )
                    )

                    should_reverse = (
                        opposite_entry
                        and cfg.allow_reversal
                    )

                    if explicit_exit or should_reverse:
                        old_side = position.side

                        if explicit_exit:
                            exit_reason = (
                                f"{old_side}_exit_signal"
                            )
                        else:
                            exit_reason = (
                                "opposite_entry_reversal"
                            )

                        closed_side = (
                            close_current_position(
                                raw_price=raw_open,
                                fill_time=bar_start,
                                fill_bar_end=bar_end,
                                fill_bar_position=(
                                    bar_position
                                ),

                                exit_signal_time=(
                                    signal_time
                                ),
                                exit_signal_bar_end=(
                                    signal_bar_end
                                ),

                                exit_reason=exit_reason,
                                decision_id=decision_id,
                                exit_context=(
                                    exit_context
                                ),
                            )
                        )

                        actions.append(
                            f"closed_{closed_side}"
                        )

                        fill_count_series[
                            bar_position
                        ] += 1

                        executed_exit_series[
                            bar_position
                        ] = True

                # -------------------------
                # 2. พิจารณา Entry ใหม่
                # -------------------------

                desired_entry: Optional[str] = None

                if long_entry and short_entry:
                    actions.append(
                        "entry_conflict_skipped"
                    )

                elif long_entry:
                    desired_entry = "long"

                elif short_entry:
                    desired_entry = "short"

                if (
                    desired_entry is not None
                    and position is None
                ):
                    entry_allowed = True

                    if closed_side is not None:
                        same_side_reentry = (
                            desired_entry
                            == closed_side
                        )

                        opposite_side_reentry = (
                            desired_entry
                            != closed_side
                        )

                        if (
                            same_side_reentry
                            and not cfg.allow_same_bar_reentry
                        ):
                            entry_allowed = False
                            actions.append(
                                "same_side_reentry_skipped"
                            )

                        if (
                            opposite_side_reentry
                            and not cfg.allow_reversal
                        ):
                            entry_allowed = False
                            actions.append(
                                "reversal_entry_skipped"
                            )

                    if entry_allowed:
                        opened, message = (
                            open_new_position(
                                side=desired_entry,
                                raw_price=raw_open,

                                fill_time=bar_start,
                                fill_bar_end=bar_end,
                                fill_bar_position=(
                                    bar_position
                                ),

                                entry_signal_time=(
                                    signal_time
                                ),
                                entry_signal_bar_end=(
                                    signal_bar_end
                                ),

                                decision_id=(
                                    decision_id
                                ),
                                entry_context=(
                                    entry_context
                                ),
                            )
                        )

                        actions.append(message)

                        if opened:
                            opened_side = (
                                desired_entry
                            )

                            fill_count_series[
                                bar_position
                            ] += 1

                            executed_entry_series[
                                bar_position
                            ] = True

                elif (
                    desired_entry is not None
                    and position is not None
                ):
                    if (
                        desired_entry
                        == position.side
                    ):
                        actions.append(
                            "same_side_entry_ignored"
                        )
                    else:
                        actions.append(
                            "opposite_entry_ignored"
                        )

                # Exit signal ขณะไม่มี Position
                if (
                    position_before == "flat"
                    and desired_entry is None
                    and (long_exit or short_exit)
                ):
                    actions.append(
                        "exit_signal_while_flat"
                    )

                if not actions:
                    actions.append(
                        "no_applicable_action"
                    )

                if (
                    closed_side is not None
                    and opened_side is not None
                    and closed_side != opened_side
                ):
                    executed_reversal_series[
                        bar_position
                    ] = True

                position_after = (
                    position.side
                    if position is not None
                    else "flat"
                )

                decision_rows.append(
                    {
                        "decision_id": (
                            decision_id
                        ),

                        "signal_time": signal_time,
                        "signal_bar_end": (
                            signal_bar_end
                        ),

                        "scheduled_fill_time": (
                            bar_start
                        ),
                        "scheduled_fill_bar_end": (
                            bar_end
                        ),

                        "long_entry_signal": (
                            long_entry
                        ),
                        "short_entry_signal": (
                            short_entry
                        ),
                        "long_exit_signal": (
                            long_exit
                        ),
                        "short_exit_signal": (
                            short_exit
                        ),

                        "position_before": (
                            position_before
                        ),
                        "position_after": (
                            position_after
                        ),

                        "outcome": "|".join(
                            actions
                        ),
                    }
                )

        # Position ที่ยังเปิดหลัง open ของ bar นี้
        # จะสัมผัส high/low ของ bar นี้
        if position is not None:
            update_position_excursions(
                position,
                bar_high=bar_high,
                bar_low=bar_low,
                bar_end=bar_end,
            )

        # ปิด Position ที่ close ของข้อมูลสุดท้าย
        if (
            bar_position == n - 1
            and cfg.close_position_at_end
            and position is not None
        ):
            close_current_position(
                raw_price=bar_close,
                fill_time=bar_end,
                fill_bar_end=bar_end,
                fill_bar_position=bar_position,

                exit_signal_time=None,
                exit_signal_bar_end=None,

                exit_reason="end_of_data",
                decision_id=None,
                exit_context=None,
                exit_at_bar_close=True,
            )

            fill_count_series[
                bar_position
            ] += 1

            executed_exit_series[
                bar_position
            ] = True

        # -------------------------
        # Equity ณ close ของ bar
        # -------------------------

        if position is None:
            unrealized_pnl = 0.0
            position_sign = 0
            position_quantity = 0.0
            position_notional = 0.0
            current_trade_id = -1

        else:
            unrealized_pnl = (
                position.sign
                * position.quantity
                * (
                    bar_close
                    - position.entry_fill_price
                )
            )

            position_sign = position.sign
            position_quantity = (
                position.quantity
            )

            position_notional = (
                position.quantity
                * bar_close
            )

            current_trade_id = (
                position.trade_id
            )

        mtm_equity = (
            cash + unrealized_pnl
        )

        cash_series[bar_position] = cash

        realized_equity_series[
            bar_position
        ] = cash

        unrealized_pnl_series[
            bar_position
        ] = unrealized_pnl

        mtm_equity_series[
            bar_position
        ] = mtm_equity

        position_sign_series[
            bar_position
        ] = position_sign

        position_qty_series[
            bar_position
        ] = position_quantity

        position_notional_series[
            bar_position
        ] = position_notional

        trade_id_series[
            bar_position
        ] = current_trade_id

    # Signal ช่วงท้ายที่ไม่มี bar อนาคตให้ Fill
    first_unfillable = max(
        0,
        n - cfg.execution_delay_bars,
    )

    for source_position in range(
        first_unfillable,
        n,
    ):
        signal_row = d.iloc[
            source_position
        ]

        signals = {
            "long_entry_signal": bool(
                signal_row[
                    "filtered_long_entry_signal"
                ]
            ),
            "short_entry_signal": bool(
                signal_row[
                    "filtered_short_entry_signal"
                ]
            ),
            "long_exit_signal": bool(
                signal_row[
                    "filtered_long_exit_signal"
                ]
            ),
            "short_exit_signal": bool(
                signal_row[
                    "filtered_short_exit_signal"
                ]
            ),
        }

        if not any(signals.values()):
            continue

        decision_rows.append(
            {
                "decision_id": (
                    next_decision_id
                ),

                "signal_time": pd.Timestamp(
                    signal_row["signal_time"]
                ),
                "signal_bar_end": pd.Timestamp(
                    d.index[source_position]
                ),

                "scheduled_fill_time": pd.NaT,
                "scheduled_fill_bar_end": (
                    pd.NaT
                ),

                **signals,

                "position_before": None,
                "position_after": None,

                "outcome": (
                    "no_future_bar_for_fill"
                ),
            }
        )

        next_decision_id += 1

    d["cash"] = cash_series

    d["realized_equity"] = (
        realized_equity_series
    )

    d["unrealized_pnl"] = (
        unrealized_pnl_series
    )

    d["mtm_equity"] = mtm_equity_series

    d["position_sign"] = (
        position_sign_series
    )

    d["position_quantity"] = (
        position_qty_series
    )

    d["position_notional"] = (
        position_notional_series
    )

    d["gross_exposure"] = np.where(
        d["mtm_equity"] != 0,
        (
            d["position_notional"].abs()
            / d["mtm_equity"].abs()
        ),
        np.nan,
    )

    d["active_trade_id"] = pd.Series(
        trade_id_series,
        index=d.index,
        dtype="Int64",
    ).where(
        trade_id_series >= 0
    )

    d["fill_count"] = fill_count_series

    d["executed_entry"] = (
        executed_entry_series
    )

    d["executed_exit"] = (
        executed_exit_series
    )

    d["executed_reversal"] = (
        executed_reversal_series
    )

    d["mtm_return"] = (
        d["mtm_equity"]
        .pct_change()
        .fillna(0.0)
    )

    d["realized_equity_change"] = (
        d["realized_equity"]
        .diff()
        .fillna(
            d["realized_equity"]
            - cfg.initial_capital
        )
    )

    decisions = pd.DataFrame(
        decision_rows
    )

    fills = pd.DataFrame(
        fill_rows
    )

    trades = pd.DataFrame(
        trade_rows
    )

    open_position_summary: Optional[
        dict[str, object]
    ] = None

    if position is not None:
        final_close = float(
            d["close"].iloc[-1]
        )

        final_unrealized = (
            position.sign
            * position.quantity
            * (
                final_close
                - position.entry_fill_price
            )
        )

        open_position_summary = {
            "trade_id": position.trade_id,
            "side": position.side,
            "quantity": position.quantity,

            "entry_signal_time": (
                position.entry_signal_time
            ),
            "entry_fill_time": (
                position.entry_fill_time
            ),

            "entry_fill_price": (
                position.entry_fill_price
            ),
            "entry_notional": (
                position.entry_notional
            ),

            "last_close": final_close,
            "unrealized_pnl": (
                final_unrealized
            ),

            "mfe_pnl": position.mfe_pnl,
            "mae_pnl": position.mae_pnl,
            "mfe_time": position.mfe_time,
            "mae_time": position.mae_time,
        }

        open_position_summary.update(
            position.entry_context
        )

    return ExecutionResult(
        strategy_id=(
            filtered_result.filtered_strategy_id
        ),
        execution_timeframe=(
            execution_timeframe
        ),
        config=cfg,

        bars=d,
        decisions=decisions,
        fills=fills,
        trades=trades,

        open_position=open_position_summary,
    )

In [103]:
def validate_execution_result(
    result: ExecutionResult,
) -> None:
    bars = result.bars
    fills = result.fills
    trades = result.trades

    required_bar_columns = {
        "cash",
        "realized_equity",
        "unrealized_pnl",
        "mtm_equity",

        "position_sign",
        "position_quantity",
        "position_notional",

        "executed_entry",
        "executed_exit",
        "executed_reversal",
    }

    missing = required_bar_columns.difference(
        bars.columns
    )

    if missing:
        raise AssertionError(
            f"Missing execution bar columns: "
            f"{sorted(missing)}"
        )

    expected_equity = (
        bars["cash"]
        + bars["unrealized_pnl"]
    )

    if not np.allclose(
        bars["mtm_equity"],
        expected_equity,
        equal_nan=True,
    ):
        raise AssertionError(
            "MTM equity != cash + unrealized PnL."
        )

    if not bars["position_sign"].isin(
        [-1, 0, 1]
    ).all():
        raise AssertionError(
            "Invalid position sign."
        )

    flat_with_quantity = (
        bars["position_sign"].eq(0)
        & bars["position_quantity"].ne(0)
    )

    if flat_with_quantity.any():
        raise AssertionError(
            "Flat Position has non-zero quantity."
        )

    if not fills.empty:
        if fills["fill_id"].duplicated().any():
            raise AssertionError(
                "Duplicated fill IDs."
            )

        if (fills["quantity"] <= 0).any():
            raise AssertionError(
                "Fill quantity must be positive."
            )

        if (fills["fee"] < 0).any():
            raise AssertionError(
                "Negative fee detected."
            )

        if (
            fills["slippage_cost"] < 0
        ).any():
            raise AssertionError(
                "Negative slippage cost detected."
            )

        fills_with_signal = fills[
            "signal_time"
        ].notna()

        fill_before_signal = (
            fills_with_signal
            & (
                pd.to_datetime(
                    fills["fill_time"],
                    utc=True,
                )
                <
                pd.to_datetime(
                    fills["signal_time"],
                    utc=True,
                )
            )
        )

        if fill_before_signal.any():
            raise AssertionError(
                "Fill occurred before signal time."
            )

    if not trades.empty:
        if trades["trade_id"].duplicated().any():
            raise AssertionError(
                "Duplicated trade IDs."
            )

        if not trades["side"].isin(
            ["long", "short"]
        ).all():
            raise AssertionError(
                "Invalid trade side."
            )

        if (
            trades["exit_fill_time"]
            < trades["entry_fill_time"]
        ).any():
            raise AssertionError(
                "Trade exited before entry."
            )

        expected_net = (
            trades[
                "gross_pnl_after_slippage"
            ]
            - trades["total_fees"]
        )

        if not np.allclose(
            trades["net_pnl"],
            expected_net,
            equal_nan=True,
        ):
            raise AssertionError(
                "Trade net PnL is inconsistent."
            )

        if (
            trades["holding_bars"] < 0
        ).any():
            raise AssertionError(
                "Negative holding bars."
            )

        ordered = trades.sort_values(
            "entry_fill_time"
        )

        previous_exit = (
            ordered["exit_fill_time"]
            .shift(1)
        )

        overlapping = (
            previous_exit.notna()
            & (
                ordered["entry_fill_time"]
                < previous_exit
            )
        )

        if overlapping.any():
            raise AssertionError(
                "Overlapping trades detected."
            )

    if result.open_position is None:
        final_position = int(
            bars["position_sign"].iloc[-1]
        )

        if final_position != 0:
            raise AssertionError(
                "No open_position summary, "
                "but final Position is not flat."
            )

รัน Baseline และ Regime-Filtered Strategy

Breakout Baseline

Breakout + 4h Regime Filter

Mean Reversion Baseline

Mean Reversion + Regime Filter

In [104]:
def summarize_execution_result(
    result: ExecutionResult,
) -> pd.Series:
    bars = result.bars
    trades = result.trades
    fills = result.fills

    initial_capital = (
        result.config.initial_capital
    )

    final_mtm_equity = float(
        bars["mtm_equity"].iloc[-1]
    )

    final_realized_equity = float(
        bars["realized_equity"].iloc[-1]
    )

    if trades.empty:
        winning_trades = 0
        losing_trades = 0
        win_rate = np.nan
        total_net_pnl = 0.0
        mean_trade_pnl = np.nan
    else:
        winning_trades = int(
            (trades["net_pnl"] > 0).sum()
        )

        losing_trades = int(
            (trades["net_pnl"] < 0).sum()
        )

        win_rate = float(
            (trades["net_pnl"] > 0).mean()
        )

        total_net_pnl = float(
            trades["net_pnl"].sum()
        )

        mean_trade_pnl = float(
            trades["net_pnl"].mean()
        )

    return pd.Series(
        {
            "strategy_id": (
                result.strategy_id
            ),

            "n_bars": len(bars),
            "n_decisions": len(
                result.decisions
            ),
            "n_fills": len(fills),
            "n_closed_trades": len(trades),

            "open_position": (
                result.open_position is not None
            ),

            "initial_capital": (
                initial_capital
            ),
            "final_realized_equity": (
                final_realized_equity
            ),
            "final_mtm_equity": (
                final_mtm_equity
            ),

            "mtm_total_return": (
                final_mtm_equity
                / initial_capital
                - 1.0
            ),

            "total_net_trade_pnl": (
                total_net_pnl
            ),
            "mean_trade_pnl": (
                mean_trade_pnl
            ),

            "winning_trades": (
                winning_trades
            ),
            "losing_trades": (
                losing_trades
            ),
            "win_rate": win_rate,

            "total_fees": (
                float(fills["fee"].sum())
                if not fills.empty
                else 0.0
            ),

            "total_slippage_cost": (
                float(
                    fills[
                        "slippage_cost"
                    ].sum()
                )
                if not fills.empty
                else 0.0
            ),
        }
    )

In [105]:
# summarize_execution_result(
#             breakout_baseline_execution
#         )

In [106]:
# summarize_execution_result(
#             breakout_4h_execution
#         )

In [107]:
# execution_summary = pd.DataFrame(
#     [
#         summarize_execution_result(
#             breakout_baseline_execution
#         ),

#         summarize_execution_result(
#             breakout_4h_execution
#         ),

#         summarize_execution_result(
#             mean_reversion_baseline_execution
#         ),

#         summarize_execution_result(
#             mean_reversion_4h_flat_execution
#         ),
#     ]
# )

# display(execution_summary)

In [108]:
# display(
#     breakout_4h_execution.trades[
#         [
#             "trade_id",
#             "side",

#             "entry_signal_time",
#             "entry_fill_time",
#             "exit_fill_time",

#             "entry_fill_price",
#             "exit_fill_price",

#             "quantity",
#             "entry_notional",

#             "net_pnl",
#             "net_return_on_entry",

#             "holding_bars",
#             "holding_duration",

#             "mfe_pnl",
#             "mae_pnl",
#             "mfe_return_on_entry",
#             "mae_return_on_entry",

#             "exit_reason",
#         ]
#     ].head(50)
# )

In [109]:
# state_trade_columns = [
#     col
#     for col
#     in breakout_4h_execution.trades.columns
#     if "confirmed_state" in col
# ]

# display(
#     breakout_4h_execution.trades[
#         [
#             "trade_id",
#             "side",
#             "entry_signal_time",
#             "net_pnl",
#         ]
#         + state_trade_columns
#     ].head(50)
# )

In [110]:
# display(
#     breakout_4h_execution.fills[
#         [
#             "fill_id",
#             "decision_id",
#             "trade_id",

#             "fill_action",
#             "position_side",
#             "transaction_side",

#             "signal_time",
#             "fill_time",
#             "fill_bar_end",

#             "raw_price",
#             "fill_price",

#             "quantity",
#             "notional",

#             "fee",
#             "slippage_cost",

#             "cash_after_fill",
#             "exit_reason",
#         ]
#     ].head(100)
# )

In [111]:
# display(
#     breakout_4h_execution.decisions[
#         [
#             "decision_id",

#             "signal_time",
#             "scheduled_fill_time",

#             "long_entry_signal",
#             "short_entry_signal",
#             "long_exit_signal",
#             "short_exit_signal",

#             "position_before",
#             "position_after",

#             "outcome",
#         ]
#     ].head(100)
# )